# Agent 4: Time Series Forecasting

## Context
Capital planning requires looking into the future. The **Forecasting Agent** uses historical monthly data to predict future claim volumes and premium income, enabling the business to maintain adequate liquidity.

## Working Logic
1. **Time Series Aggregation:** Groups the transactional data by `Date` to create a monthly frequency time series of premiums and claims.
2. **Prophet Modeling:** Attempts to train a Facebook Prophet model to capture complex trends and yearly seasonality. Includes `Inflation_Rate` as a regressor.
3. **Linear Fallback:** If the dataset is too small for Prophet, the agent gracefully falls back to a deterministic Linear Trend model with seasonality factors.
4. **Prediction:** Generates a forecast horizon (configurable, default 12 months).

## Key Concepts
- **Facebook Prophet:** An open-source additive regression model designed for analyzing time series.
- **Seasonality:** Periodic fluctuations in time series data.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Load data
try:
    df = pd.read_excel('Dataset/Cleaned.xlsx')
except:
    # Dummy data if dataset not present
    dates = pd.date_range("2022-01-01", periods=24, freq="MS")
    df = pd.DataFrame({
        "Date": dates,
        "Written_Premium": np.random.uniform(10000, 20000, 24),
        "Claim_Amount": np.random.uniform(5000, 15000, 24),
        "Inflation_Rate": np.random.uniform(1.0, 5.0, 24),
        "Customer_Segment": np.random.choice(["Retail", "Commercial", "SME"], 24),
        "Distribution_Channel": np.random.choice(["Broker", "Direct", "Agency"], 24)
    })
print("Data loaded with shape:", df.shape)

### 1. Data Preparation & Monthly Aggregation

In [ ]:
# Clean and prepare columns
df["_Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["_Premium"] = df.get("Written_Premium", 0).fillna(0)
df["_Claims"] = df.get("Claim_Amount", 0).fillna(0)
df["_Inflation"] = df.get("Inflation_Rate", 0).fillna(0)

# Aggregate to monthly
df["_Month"] = df["_Date"].dt.to_period("M").dt.to_timestamp()
monthly = df.groupby("_Month").agg(
    Written_Premium=("_Premium", "sum"),
    Claim_Amount=("_Claims", "sum"),
    Inflation_Rate=("_Inflation", "mean")
).reset_index().rename(columns={"_Month": "Month"})

monthly.head()

### 2. Prophet Forecasting (with Inflation Regressor)

In [ ]:
from prophet import Prophet

# Prepare for Prophet (ds, y)
df_ts = monthly[["Month", "Claim_Amount"]].copy()
df_ts.columns = ["ds", "y"]

m = Prophet(seasonality_mode='multiplicative')
has_inflation = "Inflation_Rate" in monthly.columns and monthly["Inflation_Rate"].sum() != 0

if has_inflation:
    m.add_regressor('Inflation_Rate')
    df_ts['Inflation_Rate'] = monthly['Inflation_Rate']

m.fit(df_ts)

# Predict future 12 months
future = m.make_future_dataframe(periods=12, freq="MS")
if has_inflation:
    future['Inflation_Rate'] = df_ts['Inflation_Rate'].iloc[-1] # Assume last known inflation

forecast = m.predict(future)
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

### 3. Segment and Channel Analysis

In [ ]:
# Customer Segment Breakdown
segment_df = df.groupby("Customer_Segment").agg(
    Total_Premium=("Written_Premium", "sum"),
    Total_Claims=("Claim_Amount", "sum")
).reset_index()
segment_df["Loss_Ratio"] = (segment_df["Total_Claims"] / segment_df["Total_Premium"] * 100).round(2)

print("Segment Breakdown:\n", segment_df.sort_values("Total_Claims", ascending=False))